In [1]:
# 1. IMPORTS E CONFIGURAÇÕES
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py
import scipy.io as sio
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# --- ESTILO TESE (ABNT/IEEE) ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'savefig.dpi': 600,
    'lines.linewidth': 2,
    'axes.grid': True,
    'grid.alpha': 0.3
})

# Paleta de Cores Consistente
CORES = {
    'MRN': '#1f77b4',  # Azul
    'T2F': '#ff7f0e',  # Laranja
    'ComReg': '#2ca02c',  # Verde
    'SemReg': '#d62728',  # Vermelho
    'ComTerra': '#9467bd',  # Roxo
    'SemTerra': '#8c564b',  # Marrom
    'FaseA': '#0000FF',
    'FaseB': '#FF0000',
    'FaseC': '#00FF00'
}

print("✅ Ambiente configurado para Tese.")

✅ Ambiente configurado para Tese.


In [2]:
# 2. MOTOR DE PROCESSAMENTO DE SINAIS
class ProcessadorSinais:
    def __init__(self, t, V, I, nome_arquivo, freq=60):
        self.t = t
        self.v = V
        self.i = I
        self.nome_arquivo = nome_arquivo
        self.freq = freq
        self.fs = 1.0 / (t[1] - t[0]) if len(t) > 1 else 60 * 256

        # Metadata extraído do nome (ajuste conforme necessidade)
        nome = nome_arquivo.lower()
        self.topo = 'T2F' if 't2f' in nome else 'MRN'
        self.reg = 'ComReg' if ('com_reg' in nome or 'comreg' in nome) else 'SemReg'
        self.terra = 'SemTerra' if ('sem_terra' in nome or 'semterra' in nome) else 'ComTerra'
        self.falta = 'ABC'  # Default, pode ser refinado
        if 'ag' in nome or 'a-g' in nome:
            self.falta = 'AG'
        elif 'bc' in nome:
            self.falta = 'BC'

    def get_rms(self, sinal):
        """Calcula RMS móvel (janela de 1 ciclo)."""
        window = int(self.fs / self.freq)
        return np.sqrt(uniform_filter1d(sinal ** 2, size=window, axis=0))

    def get_phasors(self, t_janela_inicio, n_cycles=1):
        """Retorna fasores (magnitude e ângulo) para um instante."""
        idx = np.searchsorted(self.t, t_janela_inicio)
        window = int((self.fs / self.freq) * n_cycles)

        phasors = []
        for fase in range(3):
            segmento = self.v[idx:idx + window, fase]
            fft_res = np.fft.rfft(segmento)
            # A fundamental é o segundo bin (o primeiro é DC) se n_cycles=1, mas vamos achar o pico
            freqs = np.fft.rfftfreq(len(segmento), 1 / self.fs)
            idx_60 = np.argmin(np.abs(freqs - 60))
            phasors.append(fft_res[idx_60])  # Valor complexo

        return np.array(phasors)

    def get_sym_components(self, t_instante):
        """Retorna V0, V1, V2 (Zero, Positiva, Negativa)."""
        phasors = self.get_phasors(t_instante)
        a = np.exp(1j * 2 * np.pi / 3)
        A_mat = np.array([[1, 1, 1], [1, a ** 2, a], [1, a, a ** 2]]) / 3
        seq = A_mat @ phasors
        return np.abs(seq)  # Retorna magnitudes [Zero, Pos, Neg]

    def get_thd_harmonics(self, sinal, t_inicio, t_fim):
        """Calcula THD, Fundamental e 3ª Harmônica."""
        idx_ini = np.searchsorted(self.t, t_inicio)
        idx_fim = np.searchsorted(self.t, t_fim)
        segmento = sinal[idx_ini:idx_fim]

        # Janela de Hanning para suavizar
        window = np.hanning(len(segmento))
        fft_vals = np.abs(np.fft.rfft(segmento * window)) * 2 / np.sum(window)
        freqs = np.fft.rfftfreq(len(segmento), 1 / self.fs)

        idx_60 = np.argmin(np.abs(freqs - 60))
        idx_180 = np.argmin(np.abs(freqs - 180))

        v1 = fft_vals[idx_60]
        v3 = fft_vals[idx_180]

        # THD (considerando até a 40ª)
        harmonics_sum = np.sum(fft_vals[idx_60 + 1:] ** 2)  # Simplificado
        thd = (np.sqrt(harmonics_sum) / v1) * 100 if v1 > 0 else 0

        return v1, v3, thd


# Função auxiliar de carregamento (mantida simples)
def carregar_arquivo(path):
    try:
        with h5py.File(path, 'r') as f:
            # Lógica simplificada para HDF5 v7.3
            t = f[list(f.keys())[0]]  # Pega o primeiro dataset como exemplo (ajustar)
            # ... (Implementar lógica de extração igual ao script anterior)
            pass
    except:
        try:
            mat = sio.loadmat(path)
            # Procura chaves
            keys = [k for k in mat.keys() if '_' in k and 'raw' not in k]
            # Assumindo estrutura padrão
            # Esta parte precisa ser adaptada ao seu formato exato de saída do MATLAB
            # Simulação de estrutura para o exemplo:
            t = mat.get('t', mat.get('time', np.array([]))).flatten()

            # Tenta achar V e I baseados no nome do arquivo ou varredura
            # Aqui faremos uma busca genérica
            V = None
            I = None
            for k in mat.keys():
                if k.startswith('V_') and k.endswith('_raw') == False: V = mat[k]
                if k.startswith('I_') and k.endswith('_raw') == False: I = mat[k]

            # Fallback para arrays vazios se falhar
            if V is None: V = np.zeros((len(t), 3))
            if I is None: I = np.zeros((len(t), 3))

            # Correção de shape (N, 3)
            if V.shape[0] == 3: V = V.T
            if I.shape[0] == 3: I = I.T

            return ProcessadorSinais(t, V, I, Path(path).name)
        except Exception as e:
            print(f"Erro ao ler {path}: {e}")
            return None


print("✅ Classes definidas.")

✅ Classes definidas.


In [3]:
# 3. ORGANIZAÇÃO DOS CASOS (Banco de Dados)

PASTA_RAIZ = (
    "C:/Users/Leonardo Felipe/OneDrive/Coisas_Leonardo/gits/CurtosT2F/"
    "T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/"
    "Teste_Novo_Sem_Terra_14/Processados_HDF5/"
)  # <--- AJUSTE AQUI


PASTA_SALVAR = './Resultados_analise_14'
# Dicionário para armazenar todos os processadores
# Estrutura: BD[Topologia][Regulador][Aterramento][Barra]
BD = {
    'MRN': {'ComReg': {'ComTerra': {}, 'SemTerra': {}}, 'SemReg': {'ComTerra': {}, 'SemTerra': {}}},
    'T2F': {'ComReg': {'ComTerra': {}, 'SemTerra': {}}, 'SemReg': {'ComTerra': {}, 'SemTerra': {}}}
}


def scanear_pasta(pasta):
    arquivos = list(Path(pasta).glob("*.mat"))
    print(f"🔍 Encontrados {len(arquivos)} arquivos.")

    for arq in arquivos:
        nome = arq.name.lower()

        # 1. Identificar Topologia
        topo = 'T2F' if 't2f' in nome else 'MRN'

        # 2. Identificar Regulador (Assumindo que nomes tem indicação, senão ajustar)
        # Exemplo: Se o arquivo não diz 'SemReg', assumimos que é o caso padrão?
        # Ou você tem pastas separadas?
        # AQUI VOCÊ PRECISA AJUSTAR AS STRINGS DE BUSCA
        reg = 'SemReg' if ('sem_reg' in nome or 'semreg' in nome or '_sr_' in nome) else 'ComReg'

        # 3. Identificar Aterramento
        terra = 'SemTerra' if ('sem_terra' in nome or 'semterra' in nome) else 'ComTerra'

        # 4. Identificar Barra (do nome do arquivo)
        barra = 'Desconhecida'
        for b in ['800', '816', '820', '822']:
            if b in nome:
                barra = b
                break

        # Carregar e Armazenar
        # Nota: carregar_mat deve ser a função robusta do script anterior
        # Estou usando um placeholder aqui. USE A FUNÇÃO carregar_mat DO SCRIPT ANTERIOR
        dados = carregar_arquivo(arq)

        if dados:
            BD[topo][reg][terra][barra] = dados
            print(f"  📂 Mapeado: {topo} | {reg} | {terra} | Barra {barra} -> {arq.name}")


# scanear_pasta(PASTA_RAIZ) # Descomente para rodar
print("ℹ️ Execute scanear_pasta() após configurar o caminho correto.")

ℹ️ Execute scanear_pasta() após configurar o caminho correto.


In [4]:
# 4. PLOTAGEM SEÇÃO 1: EFEITO DO ATERRAMENTO

def plot_secao_1_aterramento(BD, barra_alvo='820', salvar_dir=PASTA_SALVAR):
    Path(salvar_dir).mkdir(exist_ok=True)

    # Para cada topologia (MRN e T2F), criar uma figura
    for topo in ['MRN', 'T2F']:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
        fig.suptitle(f'Perfil de Tensão RMS - {topo} - Barra {barra_alvo}', fontsize=16)

        cenarios = [
            (0, 0, 'ComReg', 'ComTerra', 'Com Regulador / Com Terra'),
            (0, 1, 'ComReg', 'SemTerra', 'Com Regulador / Sem Terra'),
            (1, 0, 'SemReg', 'ComTerra', 'Sem Regulador / Com Terra'),
            (1, 1, 'SemReg', 'SemTerra', 'Sem Regulador / Sem Terra')
        ]

        for r, c, reg, terra, titulo in cenarios:
            ax = axes[r, c]
            try:
                proc = BD[topo][reg][terra].get(barra_alvo)
                if proc:
                    rms = proc.get_rms(proc.v)
                    ax.plot(proc.t, rms[:, 0], label='Fase A', color=CORES['FaseA'])
                    ax.plot(proc.t, rms[:, 1], label='Fase B', color=CORES['FaseB'])
                    ax.plot(proc.t, rms[:, 2], label='Fase C', color=CORES['FaseC'])
                    ax.set_title(titulo)
                    ax.grid(True, alpha=0.3)
                    if r == 1: ax.set_xlabel('Tempo (s)')
                    if c == 0: ax.set_ylabel('Tensão (V)')
                else:
                    ax.text(0.5, 0.5, 'Dados não encontrados', ha='center')
            except Exception as e:
                print(f"Erro plotando {topo} {reg} {terra}: {e}")

        # Legenda única
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, 0.02))
        plt.tight_layout(rect=[0, 0.05, 1, 0.95])
        plt.savefig(f"{salvar_dir}/Sec1_Aterramento_{topo}_Barra_{barra_alvo}.png")
        #plt.show()

# plot_secao_1_aterramento(BD, '820')

In [5]:
# 5. ANÁLISE SEÇÃO 2: 3ª HARMÔNICA (FFT)

def analise_secao_2_harmonicos(BD, barra_alvo='820', t_pre=0.1, t_falta=0.5/3, salvar_dir=PASTA_SALVAR):
    dados_tabela = []

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    fig.suptitle(f'Espectro Harmônico (Tensão) - Barra {barra_alvo}', fontsize=16)

    configs_plot = [
        ('MRN', 'ComReg', axes[0, 0]), ('T2F', 'ComReg', axes[0, 1]),
        ('MRN', 'SemReg', axes[1, 0]), ('T2F', 'SemReg', axes[1, 1])
    ]

    for topo, reg, ax in configs_plot:
        # Comparar Com Terra vs Sem Terra no mesmo gráfico
        for terra, estilo in [('ComTerra', '-'), ('SemTerra', '--')]:
            proc = BD[topo][reg][terra].get(barra_alvo)
            if proc:
                # FFT da Fase A (ou a fase faltosa)
                v1, v3, thd = proc.get_thd_harmonics(proc.v[:, 0], t_falta, t_falta + 0.05)  # Janela curta na falta

                # Guarda na tabela
                dados_tabela.append({
                    'Topologia': topo, 'Regulador': reg, 'Aterramento': terra,
                    'Barra': barra_alvo, 'Condicao': 'Falta',
                    '|V1| (V)': f"{v1:.1f}", '|V3| (V)': f"{v3:.1f}", 'THD (%)': f"{thd:.2f}"
                })

                # Plota Espectro
                freqs, mag = proc.fft_espectro(0, n_cycles=3)  # Usar método da classe original
                mask = freqs <= 300  # Zoom até 5ª harmonica
                ax.plot(freqs[mask], mag[mask], linestyle=estilo, label=f'{terra}', linewidth=1.5)

        ax.set_title(f'{topo} - {reg}')
        ax.set_ylabel('Magnitude (V)')
        ax.set_xlabel('Frequência (Hz)')
        ax.legend()
        ax.grid(True, alpha=0.3)

    # Salva Gráfico
    plt.savefig(f"{salvar_dir}/Sec2_FFT_Harmonicos_{barra_alvo}.png")
    #plt.show()

    # Salva Tabela CSV
    df = pd.DataFrame(dados_tabela)
    df.to_csv(f"{salvar_dir}/Sec2_Tabela_Harmonicos_{barra_alvo}.csv", index=False)
    print("\n📊 Tabela de Harmônicas Gerada:")
    print(df)

# analise_secao_2_harmonicos(BD, '820')

In [6]:
# 6. ANÁLISE SEÇÃO 3: COMPARAÇÃO PROTEÇÃO E QUALIDADE

def analise_secao_3_comparativo(BD, barra_protecao='816', barra_qualidade='822', t_falta=0.35,
                                salvar_dir=PASTA_SALVAR):
    resumo = []

    # Coletar dados
    for topo in ['MRN', 'T2F']:
        for reg in ['ComReg', 'SemReg']:
            for terra in ['ComTerra', 'SemTerra']:  # Pode filtrar só ComTerra se quiser

                # 1. Corrente de Falta (Barra 816)
                proc_prot = BD[topo][reg][terra].get(barra_protecao)
                if proc_prot:
                    i_rms = proc_prot.get_rms(proc_prot.i)
                    idx_falta = np.searchsorted(proc_prot.t, t_falta)
                    i_max = np.max(i_rms[idx_falta:, :])  # Máximo após início da falta
                else:
                    i_max = 0

                # 2. Desequilíbrio V2/V1 (Barra 822)
                proc_qual = BD[topo][reg][terra].get(barra_qualidade)
                if proc_qual:
                    seq = proc_qual.get_sym_components(t_falta + 0.05)  # Estabilizado na falta
                    v1, v2 = seq[1], seq[2]
                    desequilibrio = (v2 / v1) * 100 if v1 > 0 else 0

                    # Harmonica V3/V1
                    v1_h, v3_h, _ = proc_qual.get_thd_harmonics(proc_qual.v[:, 0], t_falta, t_falta + 0.05)
                    rel_v3 = (v3_h / v1_h) * 100 if v1_h > 0 else 0
                else:
                    desequilibrio = 0
                    rel_v3 = 0

                resumo.append({
                    'Topologia': topo, 'Regulador': reg, 'Aterramento': terra,
                    'I_Falta_Max': i_max, 'V2/V1 (%)': desequilibrio, 'V3/V1 (%)': rel_v3
                })

    df = pd.DataFrame(resumo)

    # --- GRÁFICO 1: Corrente de Falta ---
    fig, ax = plt.subplots(figsize=(10, 6))
    df_com_terra = df[df['Aterramento'] == 'ComTerra']

    x = np.arange(len(df_com_terra))
    width = 0.35

    # Separar MRN e T2F para cores
    # Logica de plotagem simplificada: Agrupar por (Topo + Reg)
    labels = df_com_terra.apply(lambda row: f"{row['Topologia']}\n{row['Regulador']}", axis=1)
    ax.bar(labels, df_com_terra['I_Falta_Max'], color=[CORES[r['Topologia']] for _, r in df_com_terra.iterrows()])

    ax.set_ylabel('Corrente de Falta Máxima (A)')
    ax.set_title(f'Sensibilidade da Proteção (Barra {barra_protecao}) - Com Aterramento')
    plt.savefig(f"{salvar_dir}/Sec3_Barra_Corrente_{barra_protecao}.png")
    #plt.show()

    # --- GRÁFICO 2: Desequilíbrio ---
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(labels, df_com_terra['V2/V1 (%)'], color=[CORES[r['Topologia']] for _, r in df_com_terra.iterrows()])
    ax.set_ylabel('Desequilíbrio V2/V1 (%)')
    ax.set_title(f'Desequilíbrio de Tensão (Barra {barra_qualidade}) - Com Aterramento')
    plt.savefig(f"{salvar_dir}/Sec3_Barra_Desequilibrio_{barra_qualidade}.png")
    #plt.show()

    # Salva Tabela Mãe
    df.to_csv(f"{salvar_dir}/Sec3_Tabela_Resumo_Geral.csv", index=False)
    print("\n📊 Tabela Resumo Comparativo Gerada:")
    print(df.head())

# analise_secao_3_comparativo(BD)

In [7]:
# 7. ANÁLISE SEÇÃO 4: PERFIL DE TENSÃO (REGULADOR)

def analise_secao_4_perfil(BD, barras_ordem=['800', '816', 'T2F', 'T2F1', '820', '822'], t_pre_falta=0.5/3, salvar_dir=PASTA_SALVAR):
    fig, ax = plt.subplots(figsize=(12, 6))

    # Eixo X
    x = np.arange(len(barras_ordem))

    cenarios = [
        ('MRN', 'ComReg', '-'), ('MRN', 'SemReg', '--'),
        ('T2F', 'ComReg', '-.'), ('T2F', 'SemReg', ':')
    ]

    for topo, reg, estilo in cenarios:
        perfis = []
        terra = 'ComTerra'  # Fixo para análise de perfil base

        for barra in barras_ordem:
            proc = BD[topo][reg][terra].get(barra)
            if proc:
                # Média do RMS no período pré-falta
                rms = proc.get_rms(proc.v)
                idx_pre = np.searchsorted(proc.t, t_pre_falta)
                val_medio = np.mean(rms[:idx_pre, :])  # Média das 3 fases e do tempo
                perfis.append(val_medio)
            else:
                perfis.append(None)

        # Plota linha se tiver dados completos
        if None not in perfis:
            ax.plot(x, perfis, linestyle=estilo, marker='o', linewidth=2,
                    label=f'{topo} - {reg}', color=CORES[topo])

    ax.set_xticks(x)
    ax.set_xticklabels(barras_ordem)
    ax.set_ylabel('Tensão Média Pré-Falta (V)')
    ax.set_xlabel('Barra')
    ax.set_title('Perfil de Tensão ao Longo do Alimentador (Com vs Sem Regulador)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.savefig(f"{salvar_dir}/Sec4_Perfil_Tensao.png")
    #plt.show()

# analise_secao_4_perfil(BD)